# Squad-Level Features from Club Statistics

**Goal:** Aggregate player-level club stats to one row per team 


In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "../../data"
PLAYER_PATH = f"{DATA_DIR}/processed/player_stats_with_club_elo.csv"
PROCESSED_PATH = f"{DATA_DIR}/processed/class_a_with_rankings.csv"
OUTPUT_SQUAD = f"{DATA_DIR}/processed/squad_level_features.csv"
OUTPUT_FULL = f"{DATA_DIR}/processed/class_a_with_squad_features.csv"

df = pd.read_csv(PLAYER_PATH)

# Elite threshold = top 25% of club_strength distribution
q75 = df[df["perf_found"] == True]["club_strength"].quantile(0.75)
df["is_elite_club"] = (df["club_strength"] >= q75).astype(float)

print(f"Player records:{len(df)}")
print(f"With club data: {df['perf_found'].sum()} ({df['perf_found'].mean():.1%})")
print(
    f"With club strength Elo: {df['club_strength'].notna().sum()} ({df['club_strength'].notna().mean():.1%})"
)
print(f"Elite club threshold: Elo ≥ {q75:.0f} (top 25%)")

## Aggregate to Squad Level

In [ ]:
df_found = df[df["perf_found"] == True].copy()
df_elo = df[df["club_strength"].notna()].copy()

GROUP = ["tournament_id", "team_id", "team_name"]

# Coverage
coverage = (
    df.groupby(GROUP)
    .agg(
        squad_size=("player_id", "count"),
        squad_perf_coverage=("perf_found", "mean"),
        squad_elo_coverage=("club_strength", lambda x: x.notna().mean()),
    )
    .reset_index()
)

# Club strength (data-driven, from Club Elo)
strength_feats = (
    df_elo.groupby(GROUP)
    .agg(
        squad_avg_club_strength=("club_strength", "mean"),
        squad_max_club_strength=("club_strength", "max"),
        squad_min_club_strength=("club_strength", "min"),
        squad_pct_elite_clubs=("is_elite_club", "mean"),
        squad_n_clubs=("club_name", "nunique"),
        squad_n_leagues=("competition_name", "nunique"),
    )
    .reset_index()
)

# Overall club form
form_feats = (
    df_found.groupby(GROUP)
    .agg(
        squad_avg_minutes=("minutes_played", "mean"),
        squad_total_minutes=("minutes_played", "sum"),
        squad_avg_goals=("club_goals", "mean"),
        squad_total_goals=("club_goals", "sum"),
        squad_avg_assists=("assists", "mean"),
        squad_avg_appearances=("appearances", "mean"),
    )
    .reset_index()
)

# Position-specific
fwd = df_found[df_found["position_code"] == "FW"]
mid = df_found[df_found["position_code"] == "MF"]
def_ = df_found[df_found["position_code"] == "DF"]
gk = df_found[df_found["position_code"] == "GK"]

fwd_feats = (
    fwd.groupby(GROUP)
    .agg(
        squad_fwd_avg_goals=("club_goals", "mean"),
        squad_fwd_avg_assists=("assists", "mean"),
        squad_fwd_avg_minutes=("minutes_played", "mean"),
        squad_fwd_avg_strength=("club_strength", "mean"),
    )
    .reset_index()
)

mid_feats = (
    mid.groupby(GROUP)
    .agg(
        squad_mid_avg_goals=("club_goals", "mean"),
        squad_mid_avg_assists=("assists", "mean"),
        squad_mid_avg_minutes=("minutes_played", "mean"),
        squad_mid_avg_strength=("club_strength", "mean"),
    )
    .reset_index()
)

def_feats = (
    def_.groupby(GROUP)
    .agg(
        squad_def_avg_minutes=("minutes_played", "mean"),
        squad_def_avg_goals=("club_goals", "mean"),
        squad_def_avg_strength=("club_strength", "mean"),
    )
    .reset_index()
)

gk_feats = (
    gk.groupby(GROUP)
    .agg(
        squad_gk_avg_clean_sheets=("clean_sheets", "mean"),
        squad_gk_avg_minutes=("minutes_played", "mean"),
        squad_gk_avg_goals_conceded=("goals_conceded", "mean"),
        squad_gk_avg_strength=("club_strength", "mean"),
    )
    .reset_index()
)

print("Aggregation complete:")
for name, tbl in [
    ("coverage", coverage),
    ("strength_feats", strength_feats),
    ("form_feats", form_feats),
    ("fwd_feats", fwd_feats),
    ("mid_feats", mid_feats),
    ("def_feats", def_feats),
    ("gk_feats", gk_feats),
]:
    print(f"  {name}: {tbl.shape}")

## Merge All Squad Features

In [ ]:
squad = coverage.copy()
for tbl in [strength_feats, form_feats, fwd_feats, mid_feats, def_feats, gk_feats]:
    squad = squad.merge(tbl, on=GROUP, how="left")

wc_year_map = {
    "WC-2006": 2006,
    "WC-2010": 2010,
    "WC-2014": 2014,
    "WC-2018": 2018,
    "WC-2022": 2022,
}
squad["year"] = squad["tournament_id"].map(wc_year_map)

print(f"Squad features shape: {squad.shape}")
print(f"Columns: {list(squad.columns)}")

## Top Teams by Club Strength (2022 WC)

In [ ]:
wc22 = squad[squad["year"] == 2022].sort_values(
    "squad_avg_club_strength", ascending=False
)
print("Top 10 squads by avg club strength — 2022 WC:")
print(
    wc22[
        [
            "team_name",
            "squad_avg_club_strength",
            "squad_max_club_strength",
            "squad_pct_elite_clubs",
            "squad_avg_goals",
            "squad_gk_avg_clean_sheets",
        ]
    ]
    .head(10)
    .round(1)
    .to_string(index=False)
)

print()
print("Average squad club strength by tournament:")
print(
    squad.groupby("year")["squad_avg_club_strength"]
    .mean()
    .round(1)
    .rename("avg_club_strength")
    .to_string()
)

## Join onto Processed ML Dataset

In [ ]:
processed = pd.read_csv(PROCESSED_PATH)
print(f"Processed dataset shape: {processed.shape}")

squad_feat_cols = [c for c in squad.columns if c not in GROUP + ["year", "team_name"]]
squad_lookup = squad[["team_id", "year"] + squad_feat_cols].copy()

# Join home and away squad features
processed = processed.merge(
    squad_lookup.rename(
        columns={"team_id": "home_team_id", **{c: f"home_{c}" for c in squad_feat_cols}}
    ),
    on=["home_team_id", "year"],
    how="left",
)
processed = processed.merge(
    squad_lookup.rename(
        columns={"team_id": "away_team_id", **{c: f"away_{c}" for c in squad_feat_cols}}
    ),
    on=["away_team_id", "year"],
    how="left",
)

# Diff features (home minus away) — built all at once to avoid fragmentation
diff_pairs = [
    ("squad_avg_club_strength", "feat_squad_club_strength_diff"),
    ("squad_pct_elite_clubs", "feat_squad_elite_pct_diff"),
    ("squad_avg_goals", "feat_squad_goals_diff"),
    ("squad_avg_assists", "feat_squad_assists_diff"),
    ("squad_avg_minutes", "feat_squad_minutes_diff"),
    ("squad_fwd_avg_goals", "feat_squad_fwd_goals_diff"),
    ("squad_fwd_avg_strength", "feat_squad_fwd_strength_diff"),
    ("squad_gk_avg_clean_sheets", "feat_squad_gk_clean_sheets_diff"),
]

diff_cols = {
    diff_name: processed[f"home_{feat}"] - processed[f"away_{feat}"]
    for feat, diff_name in diff_pairs
    if f"home_{feat}" in processed.columns and f"away_{feat}" in processed.columns
}
processed = pd.concat(
    [processed, pd.DataFrame(diff_cols, index=processed.index)], axis=1
)

new_cols = [c for c in processed.columns if "squad" in c]
print(f"Final shape: {processed.shape}")
print(f"New squad columns: {len(new_cols)}")

## Coverage Check

In [ ]:
key_feats = [
    "home_squad_avg_club_strength",
    "home_squad_pct_elite_clubs",
    "home_squad_avg_goals",
    "home_squad_fwd_avg_goals",
    "home_squad_gk_avg_clean_sheets",
    "feat_squad_club_strength_diff",
]
print("Squad feature coverage (non-null %):\n")
print(processed[key_feats].notna().mean().mul(100).round(1).to_string())

print("\nCoverage by year (squad features covers 2006-2022):")
print(
    processed.groupby("year")["home_squad_avg_club_strength"]
    .apply(lambda x: f"{x.notna().mean():.0%}")
    .rename("coverage")
    .to_string()
)

In [ ]:
squad.to_csv(OUTPUT_SQUAD, index=False)
processed.to_csv(OUTPUT_FULL, index=False)
print(f"Squad features:    {OUTPUT_SQUAD}  {squad.shape}")
print(f"Full ML dataset:   {OUTPUT_FULL}  {processed.shape}")

In [ ]:
# Year filter
YEAR_START, YEAR_END = 2006, 2022

filtered = processed[
    (processed["year"] >= YEAR_START) & (processed["year"] <= YEAR_END)
].dropna(subset=["home_squad_avg_club_strength"])

print(f"Filtered: {filtered.shape[0]} matches ({YEAR_START}-{YEAR_END})")
print(f"Columns:  {filtered.shape[1]}")
filtered.to_csv(
    f"{DATA_DIR}/final/class_a_squad_{YEAR_START}_{YEAR_END}.csv", index=False
)
print(f"Saved: class_a_squad_{YEAR_START}_{YEAR_END}.csv")